In this notebook we look into checking if there is any benefit to combining the TF-IDF and INSTRUCTOR model skeletons we ended up with in the baseline.ipynb and transformer.ipynb notebooks. 

In [2]:
import sys
from pathlib import Path

parent_dir = str(Path().resolve().parents[1])
sys.path.insert(0, parent_dir)

In [3]:
import pandas as pd

from src.Qlassifier.evaluation import evaluate_predictions

In [27]:
instructor_df = pd.read_csv(Path("results") / "model2_chem.csv", index_col=0)
instructor_df_math = pd.read_csv(Path("results") / "model2_math.csv", index_col=0)

tfidf_df = pd.read_csv(Path("results") / "tf_idf_chem_out.csv", index_col=0)
tfidf_df_math = pd.read_csv(Path("results") / "tf_idf_math_out.csv", index_col=0) 

First things first, we need to merge our predictions.

In [42]:
instructor_df = instructor_df.rename({"confidence": "inst_conf"}, axis=1).drop(["report_pred"], axis=1) # drop report, too inconsistent
tfidf_df = tfidf_df.rename({"confidence": "tfidf_conf", "pred_topic_idx": "tfidf_pred"}, axis=1)

combined_df = pd.concat([instructor_df, tfidf_df[["tfidf_conf", "tfidf_pred"]]], axis=1)

In [43]:
combined_df.head(5)

,label,text,comments,total_marks,pred_topic,pred_topic_idx,true_topic_idx,true_topic,inst_conf,tfidf_conf,tfidf_pred
0,Question 1,Which one of the following correctly represent...,"In human cells, glucose reacts with oxygen in ...",1,Carbon-based fuels,0,0,Carbon-based fuels,0.460373,1.000000,0
1,Question 2,Which one of the following statements is corre...,Fuel cells and galvanic cells both produce hea...,1,Primary galvanic cells and fuel cells as sourc...,2,2,Primary galvanic cells and fuel cells as sourc...,0.916702,0.796580,2
2,Question 3,"Molecules X, and all have the same number of c...",The larger the number of C=C double bonds in t...,1,"Structure, nomenclature and properties of orga...",6,6,"Structure, nomenclature and properties of orga...",0.520939,0.083445,6
3,Question 4,cell can be considered secondary cell if A. it...,The polarity of the physical electrodes does n...,1,Primary galvanic cells and fuel cells as sourc...,2,5,Production of chemicals using electrolysis,0.713771,0.511240,5
4,Question 5,Consider the following statements about coenzy...,All three statements are properties of coenzym...,1,Rates of chemical reactions,3,10,Medicinal chemistry,0.510285,0.061252,10


In [44]:
instructor_df_math = instructor_df_math.rename({"confidence": "inst_conf"}, axis=1).drop(["report_pred"], axis=1)
tfidf_df_math = tfidf_df_math.rename({"confidence": "tfidf_conf", "pred_topic_idx": "tfidf_pred"}, axis=1)

combined_df_math = pd.concat([instructor_df_math, tfidf_df_math[["tfidf_conf", "tfidf_pred"]]], axis=1)

In [37]:
combined_df_math.head()

,label,text,comments,total_marks,pred_topic,pred_topic_idx,report_pred,true_topic_idx,true_topic,inst_conf,tfidf_conf,tfidf_pred
0,Question 1,Consider the following statement. ‘If my footb...,NaN,1,Discrete mathematics: Logic and proof,0,1,0,Discrete mathematics: Logic and proof,0.885443,0.312597,0
1,Question 2,The graph of ax bx has asymptotes given by and...,NaN,1,"Functions, relations and graphs",1,1,1,"Functions, relations and graphs",0.620444,0.025168,10
2,Question 3,"In the interval π π the graph of sec( ), where...",NaN,1,"Functions, relations and graphs",1,1,1,"Functions, relations and graphs",0.342832,0.373191,11
3,Question 4,"If − 1) ai where is non-zero real constant, t...",NaN,1,"Algebra, number and structure: Complex numbers",2,1,2,"Algebra, number and structure: Complex numbers",0.145890,1.000000,9
4,Question 5,Let be complex number where Re( and Im( 0. Giv...,NaN,1,"Algebra, number and structure: Complex numbers",2,1,2,"Algebra, number and structure: Complex numbers",0.557640,0.251177,2


# Chem

In [45]:
len(combined_df[
    (combined_df["pred_topic_idx"] == combined_df["true_topic_idx"]) |
    (combined_df["tfidf_pred"] == combined_df["true_topic_idx"]
)]) / len(combined_df)

0.7317073170731707